In [2]:
import pandas as pd
from tqdm.auto import tqdm

from project_package import data_collection as dc
from project_package import utility

In [4]:
df = utility.read_data('/merged/df_merged_demo')
#show all columns
pd.set_option('display.max_columns', None)
df.head()
print(len(df))

145982


Filling Missing Value by Vehicle Types


In [5]:
feature_cols = df.columns.tolist()
print(feature_cols)

['Date', 'Time', 'Booking ID', 'Booking Status', 'Customer ID', 'Vehicle Type', 'Pickup Location', 'Drop Location', 'Avg VTAT', 'Avg CTAT', 'Cancelled Rides by Customer', 'Reason for cancelling by Customer', 'Cancelled Rides by Driver', 'Driver Cancellation Reason', 'Incomplete Rides', 'Incomplete Rides Reason', 'Booking Value', 'Ride Distance', 'Driver Ratings', 'Customer Rating', 'Payment Method', 'pick_longitude', 'pick_latitude', 'pick_address', 'pick_region', 'pick_locality', 'drop_longitude', 'drop_latitude', 'drop_address', 'drop_region', 'drop_locality', 'datetime', 'date', 'hour', 'time_pickup', 'address_x', 'temperature_2m_pickup', 'relative_humidity_2m_pickup', 'dew_point_2m_pickup', 'apparent_temperature_pickup', 'precipitation_pickup', 'rain_pickup', 'snowfall_pickup', 'wind_speed_10m_pickup', 'time_drop', 'address_y', 'temperature_2m_drop', 'relative_humidity_2m_drop', 'dew_point_2m_drop', 'apparent_temperature_drop', 'precipitation_drop', 'rain_drop', 'snowfall_drop', 'w

In [7]:
veh_type = ['eBike', 'Go Sedan', 'Auto', 'Premier Sedan', 'Bike', 'Go Mini', 'Uber XL']

#proportion of missing values in Avg VTAT and Avg CTAT by Vehicle Type
df_merged=df.groupby("Vehicle Type")[["Avg VTAT", "Avg CTAT"]].apply(lambda x: x.isna().mean())
df_merged

,Avg VTAT,Avg CTAT
Vehicle Type,,
Auto,0.071562,0.322124
Bike,0.067491,0.319704
Go Mini,0.067984,0.318716
Go Sedan,0.072593,0.327293
Premier Sedan,0.070809,0.322240
Uber XL,0.071824,0.318014
eBike,0.071582,0.322313


In [10]:
def fill_missing_and_flag(df,cols_to_process):
    """
    Create missing flags and fill NaN values with mean per Vehicle Type.

    Parameters:
        df (pd.DataFrame): Input dataframe with ride booking data.

    Returns:
        pd.DataFrame: Dataframe with new _missing_flag and _fill columns.
    """

    # Create missing flags
    for col in cols_to_process:
        flag_col = f"{col.replace(' ', '')}_missing_flag"
        df[flag_col] = df[col].isna().astype(int)

    # Fill NaNs with mean grouped by Vehicle Type
    for col in cols_to_process:
        fill_col = f"{col}_fill"
        df[fill_col] = df.groupby("Vehicle Type")[col].transform(
            lambda x: x.fillna(x.mean())
        )

    return df


In [11]:
cols_to_process = ["Avg VTAT", "Avg CTAT", "Booking Value", "Ride Distance", "Driver Ratings", "Customer Rating"]
df_filled = fill_missing_and_flag(df, cols_to_process)
df_filled.shape

(145982, 89)

In [13]:
# Group by Vehicle Type and take the mean of all _fill columns
avg_by_vehicle = (
    df_filled.groupby("Vehicle Type")[[
        "Avg VTAT_fill", "Avg CTAT_fill", "Booking Value_fill",
        "Ride Distance_fill", "Driver Ratings_fill", "Customer Rating_fill"
    ]]
    .mean()
    .reset_index()
)

avg_by_vehicle

,Vehicle Type,Avg VTAT_fill,Avg CTAT_fill,Booking Value_fill,Ride Distance_fill,Driver Ratings_fill,Customer Rating_fill
0,Auto,8.446660,29.131193,506.743182,24.618638,4.231947,4.402546
1,Bike,8.506528,29.201697,510.392273,24.640380,4.229653,4.404226
2,Go Mini,8.470235,29.173641,508.023985,24.611863,4.227354,4.404971
3,Go Sedan,8.400262,29.026485,510.877155,24.566764,4.232220,4.408785
4,Premier Sedan,8.443826,29.210540,509.407194,24.646160,4.234066,4.404010
5,Uber XL,8.582085,29.232645,503.831019,24.480383,4.238222,4.404481
6,eBike,8.489832,29.148323,503.256008,24.991943,4.224941,4.404470


In [15]:
# Group by Vehicle Type and take the median of all _fill columns
median_by_vehicle = (
    df_filled.groupby("Vehicle Type")[[
        "Avg VTAT", "Avg CTAT", "Booking Value",
        "Ride Distance", "Driver Ratings", "Customer Rating"
    ]]
    .median()
    .reset_index()
)

median_by_vehicle

,Vehicle Type,Avg VTAT,Avg CTAT,Booking Value,Ride Distance,Driver Ratings,Customer Rating
0,Auto,8.2,28.7,413.0,23.630,4.3,4.4
1,Bike,8.3,28.9,417.0,23.745,4.3,4.5
2,Go Mini,8.3,28.7,413.0,23.650,4.3,4.5
3,Go Sedan,8.2,28.6,416.0,23.625,4.3,4.5
4,Premier Sedan,8.3,28.9,415.0,23.920,4.3,4.4
5,Uber XL,8.5,28.7,407.0,23.240,4.3,4.4
6,eBike,8.3,28.8,410.0,24.190,4.3,4.5


In [22]:
vehicle_type = df_filled["Vehicle Type"].unique()

Since the average value does not differ much across vehicle type, in the following normalize steps. I will do a generate normalizing and not make it depends on vehice tyoe.

In [24]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from ipywidgets import interact


import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown

# --- Function to plot histogram by vehicle type & feature ---
def plot_hist(vehicle_type, feature):
    subset = df_filled[df_filled["Vehicle Type"] == vehicle_type]
    
    plt.figure(figsize=(8, 5))
    plt.hist(subset[feature].dropna(), bins=30, color="steelblue", alpha=0.7)
    plt.title(f"{feature} - {vehicle_type}")
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.show()

# --- Interactive dropdowns ---
interact(
    plot_hist,
    vehicle_type=Dropdown(options=df_filled["Vehicle Type"].unique(), description="Vehicle"),
    feature=Dropdown(options=[
        "Avg VTAT", "Avg CTAT", "Booking Value", "Ride Distance", 
        "Driver Ratings", "Customer Rating",
        "Avg VTAT_fill", "Avg CTAT_fill", "Booking Value_fill", 
        "Ride Distance_fill", "Driver Ratings_fill", "Customer Rating_fill"], description="Feature")
)

interactive(children=(Dropdown(description='Vehicle', options=('eBike', 'Go Sedan', 'Auto', 'Premier Sedan', '…

<function __main__.plot_hist(vehicle_type, feature)>

Scaling

In [25]:
from sklearn.preprocessing import RobustScaler
import numpy as np
import pandas as pd

def scale_features(df, features, scaler=None, suffix="_scaled"):
    """
    Scale selected features using a given scaler.
    
    Parameters:
        df (pd.DataFrame): Input dataframe.
        features (list[str]): List of feature column names to scale.
        scaler: Scaler instance (default RobustScaler()).
        suffix (str): Suffix for new scaled columns.
    
    Returns:
        pd.DataFrame: Dataframe with new scaled columns added.
    """
    if scaler is None:
        scaler = RobustScaler()
    
    df_scaled = pd.DataFrame(
        scaler.fit_transform(df[features]),
        columns=[f"{col}{suffix}" for col in features],
        index=df.index
    )
    return pd.concat([df, df_scaled], axis=1)


def log_and_scale_features(df, features, scaler=None, log_suffix="_log", scale_suffix="_scaled"):
    """
    Apply log1p transform and then scale selected features.
    
    Parameters:
        df (pd.DataFrame): Input dataframe.
        features (list[str]): List of feature column names to log+scale.
        scaler: Scaler instance (default RobustScaler()).
        log_suffix (str): Suffix for log-transformed columns.
        scale_suffix (str): Suffix for scaled columns.
    
    Returns:
        pd.DataFrame: Dataframe with log and scaled columns added.
    """
    if scaler is None:
        scaler = RobustScaler()

    # Create log-transformed features
    for col in features:
        df[f"{col}{log_suffix}"] = np.log1p(df[col])

    # Scale the log-transformed features
    log_features = [f"{col}{log_suffix}" for col in features]
    df = scale_features(df, log_features, scaler=scaler, suffix=scale_suffix)
    
    return df

In [28]:

# Weather features
weather_features = [
    'temperature_2m_pickup', 'relative_humidity_2m_pickup', 'dew_point_2m_pickup',
    'apparent_temperature_pickup', 'wind_speed_10m_pickup',
    'temperature_2m_drop', 'relative_humidity_2m_drop', 'dew_point_2m_drop',
    'apparent_temperature_drop', 'wind_speed_10m_drop'
]

# Precipitation-related features
precipitation_features = [
    'precipitation_pickup', 'rain_pickup', 'snowfall_pickup',
    'precipitation_drop', 'rain_drop', 'snowfall_drop'
]

df_filled = scale_features(df_filled, weather_features)

df_filled = log_and_scale_features(df_filled, precipitation_features)

In [30]:
df_filled.shape

(145982, 111)

In [31]:
utility.save_dataframe(df_filled,file_name='ncr_ride_bookings_with_weather_filled_scaled_v2',directory='datasets/raw/merged')

Save data completed.


Save csv with less columns

In [32]:
def get_filled_scaled_subset(df, id_cols=None):
    """
    Return a shorter dataframe with only fill/scale columns + optional ID columns.

    Parameters:
        df (pd.DataFrame): Input dataframe.
        id_cols (list[str]): Columns to always keep (e.g., ['Vehicle Type', 'Booking ID']).

    Returns:
        pd.DataFrame: Subset of df with only filled and scaled features.
    """
    # collect columns ending with _fill, _scaled, or _log
    feature_cols = [c for c in df.columns if c.endswith(("_fill", "_scaled", "_log"))]

    # add id columns if provided
    if id_cols:
        feature_cols = id_cols + feature_cols

    return df[feature_cols]

In [35]:
# keep only Vehicle Type and Booking ID along with fill/scale columns

id_cols = ['Date', 'Time', 'datetime', 'hour', 'Payment Method','Booking ID', 'Booking Status', 
    'Customer ID', 'Vehicle Type', 'Pickup Location', 'Drop Location']
df_short = get_filled_scaled_subset(df_filled, id_cols)

print(df_short.head())
df_short.shape


         Date      Time             datetime  hour Payment Method  \
0  2024-03-23  12:29:38  2024-03-23 12:29:38    12            NaN   
1  2024-11-29  18:01:39  2024-11-29 18:01:39    18            UPI   
2  2024-08-23  08:56:10  2024-08-23 08:56:10     8     Debit Card   
3  2024-10-21  17:17:25  2024-10-21 17:17:25    17            UPI   
4  2024-09-16  22:08:00  2024-09-16 22:08:00    22            UPI   

     Booking ID   Booking Status   Customer ID   Vehicle Type  \
0  "CNR5884300"  No Driver Found  "CID1982111"          eBike   
1  "CNR1326809"       Incomplete  "CID4604802"       Go Sedan   
2  "CNR8494506"        Completed  "CID9202816"           Auto   
3  "CNR8906825"        Completed  "CID2610914"  Premier Sedan   
4  "CNR1950162"        Completed  "CID9933542"           Bike   

       Pickup Location      Drop Location  Avg VTAT_fill  Avg CTAT_fill  \
0          Palam Vihar            Jhilmil       8.489832      29.148323   
1        Shastri Nagar  Gurgaon Sector 56   

(145982, 39)

In [36]:
utility.save_dataframe(df_filled,file_name='ncr_ride_bookings_with_weather_filled_scaled_short_v2',directory='datasets/raw/merged')

Save data completed.
